# PCA / Análise Fatorial — criação de indicador sintético para imóveis

Objetivo: criar um **indicador sintético para critério de preço de imóveis**, usando PCA/Análise Fatorial.

Neste exemplo, o preço real do imóvel **não entra na PCA**. Ele será usado somente no final, para verificar se o ranking criado pelos fatores tem relação com o valor real do imóvel.

Colunas removidas da análise:
- `id_imovel`
- `preco_venda_rs`
- `preco_m2_rs`

Colunas categóricas também ficam fora da PCA:
- `bairro`
- `tipo`

In [ ]:
#%% Instalação dos pacotes, se necessário
# Execute esta célula somente se algum pacote não estiver instalado no seu ambiente.

# !pip install pandas numpy scipy matplotlib scikit-learn openpyxl pingouin factor_analyzer

In [ ]:
#%% Importando bibliotecas

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from scipy.stats import chi2
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["font.size"] = 10

In [ ]:
#%% Carregando o dataset sintético

arquivo = "dataset_imoveis_pca_indicador.xlsx"

imoveis = pd.read_excel(arquivo, sheet_name="Dados_Imoveis")

display(imoveis.head())
print(imoveis.info())

## 0. Verificar se as variáveis da análise são métricas

Antes de aplicar PCA, é importante garantir que as variáveis utilizadas sejam **numéricas/métricas**.  
Neste exemplo, as colunas de identificação, preço real e preço por m² foram removidas da análise. As variáveis categóricas também foram deixadas fora.

Variáveis usadas na PCA:
- `area_m2`
- `quartos`
- `banheiros`
- `vagas_garagem`
- `andar`
- `dist_metro_m`
- `idade_imovel_anos`
- `condominio_rs`
- `iptu_anual_rs`

Resumo: esta etapa evita misturar variáveis que não fazem sentido matemático na matriz de correlação.

In [ ]:
#%% 0. Seleção das variáveis métricas para a PCA

colunas_remover = ["id_imovel", "bairro", "tipo", "preco_venda_rs", "preco_m2_rs"]

imoveis_pca = imoveis.drop(columns=colunas_remover)

# Conferência dos tipos das variáveis
tipos = imoveis_pca.dtypes.astype(str).reset_index()
tipos.columns = ["variavel", "tipo"]

display(tipos)

# Plot: tipos de variáveis selecionadas
contagem_tipos = tipos["tipo"].value_counts()

plt.figure(figsize=(7, 4))
plt.bar(contagem_tipos.index, contagem_tipos.values)
plt.title("Verificação dos tipos das variáveis usadas na PCA")
plt.xlabel("Tipo da variável")
plt.ylabel("Quantidade")
plt.tight_layout()
plt.show()

# Validação simples
todas_metricas = all(pd.api.types.is_numeric_dtype(imoveis_pca[col]) for col in imoveis_pca.columns)
print(f"Todas as variáveis selecionadas são métricas? {todas_metricas}")
print(f"Quantidade de variáveis usadas na PCA: {imoveis_pca.shape[1]}")

## 1. Verificar se existem outliers

Outliers podem distorcer correlações, autovalores e fatores.  
Aqui usamos boxplots para observar valores muito afastados nas variáveis métricas.

Resumo: o objetivo não é remover automaticamente, mas entender se existem imóveis com características muito diferentes dos demais.

In [ ]:
#%% 1. Análise visual de outliers com boxplots

variaveis = imoveis_pca.columns.tolist()

n_cols = 3
n_rows = int(np.ceil(len(variaveis) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3.5 * n_rows))
axes = axes.flatten()

for i, col in enumerate(variaveis):
    axes[i].boxplot(imoveis_pca[col], vert=True)
    axes[i].set_title(col)
    axes[i].set_ylabel("Valor")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.suptitle("Boxplots das variáveis métricas", y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

# Tabela com limites IQR apenas para apoiar a leitura
outliers_resumo = []

for col in variaveis:
    q1 = imoveis_pca[col].quantile(0.25)
    q3 = imoveis_pca[col].quantile(0.75)
    iqr = q3 - q1
    lim_inf = q1 - 1.5 * iqr
    lim_sup = q3 + 1.5 * iqr
    qtd_outliers = ((imoveis_pca[col] < lim_inf) | (imoveis_pca[col] > lim_sup)).sum()
    outliers_resumo.append([col, q1, q3, lim_inf, lim_sup, qtd_outliers])

outliers_resumo = pd.DataFrame(
    outliers_resumo,
    columns=["variavel", "Q1", "Q3", "limite_inferior", "limite_superior", "qtd_outliers"]
)

display(outliers_resumo)

## 2. Correlação de Pearson

A PCA parte da relação entre as variáveis. Por isso, avaliamos a **matriz de correlação de Pearson**.

Resumo: se quase todas as correlações fossem próximas de zero, a PCA teria pouca utilidade. Se houver correlações relevantes, faz sentido tentar resumir as variáveis em fatores/componentes.

In [ ]:
#%% 2. Matriz de correlação de Pearson

corr = imoveis_pca.corr()

display(corr.round(3))

plt.figure(figsize=(9, 7))
plt.imshow(corr, vmin=-1, vmax=1)
plt.colorbar(label="Correlação de Pearson")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right")
plt.yticks(range(len(corr.index)), corr.index)
plt.title("Matriz de Correlação de Pearson")

# Anotando os valores no gráfico
for i in range(corr.shape[0]):
    for j in range(corr.shape[1]):
        plt.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## 3. Teste de esfericidade de Bartlett

O teste de Bartlett compara a matriz de correlação observada com uma matriz identidade.

Hipóteses:

\[
H_0: R = I
\]

\[
H_1: R \neq I
\]

Se o p-valor for menor que 0,05, rejeitamos \(H_0\).  
Isso indica que a matriz de correlações é diferente da identidade e que existe correlação suficiente para prosseguir com a PCA.

In [ ]:
#%% 3. Teste de esfericidade de Bartlett calculado manualmente

n = imoveis_pca.shape[0]
k = imoveis_pca.shape[1]

det_corr = np.linalg.det(corr)

qui2_bartlett = -((n - 1) - ((2 * k + 5) / 6)) * np.log(det_corr)
gl = k * (k - 1) / 2
p_value = chi2.sf(qui2_bartlett, gl)

print(f"Determinante da matriz de correlação: {det_corr:.6f}")
print(f"Qui² Bartlett: {qui2_bartlett:.4f}")
print(f"Graus de liberdade: {gl:.0f}")
print(f"p-valor: {p_value:.8f}")

# Plot: p-valor contra o limite de 0,05
plt.figure(figsize=(7, 4))
plt.bar(["p-valor Bartlett"], [p_value])
plt.axhline(0.05, linestyle="--", label="Limite 0,05")
plt.yscale("log")
plt.ylabel("p-valor em escala log")
plt.title("Teste de esfericidade de Bartlett")
plt.legend()
plt.tight_layout()
plt.show()

if p_value < 0.05:
    print("Interpretação: rejeitamos H0. Há correlação suficiente para seguir com a PCA.")
else:
    print("Interpretação: não rejeitamos H0. A PCA pode não ser adequada.")

## 4. Autovalores e critério de Kaiser

Cada autovalor indica quanta variância é explicada por cada fator/componente.

Pelo **critério de Kaiser** ou **raiz latente**, mantemos os fatores com:

\[
\lambda \geq 1
\]

Resumo: fatores com autovalor menor que 1 explicam menos variância do que uma variável original padronizada, por isso normalmente são descartados.

In [ ]:
#%% 4. PCA inicial com todas as variáveis

# Padronização das variáveis antes da PCA
scaler = StandardScaler()
X_z = scaler.fit_transform(imoveis_pca)

pca_full = PCA(n_components=imoveis_pca.shape[1])
pca_full.fit(X_z)

autovalores = pca_full.explained_variance_
variancia = pca_full.explained_variance_ratio_
variancia_acumulada = np.cumsum(variancia)

tabela_eigen = pd.DataFrame({
    "Fator": [f"Fator {i+1}" for i in range(len(autovalores))],
    "Autovalor": autovalores,
    "Variância": variancia,
    "Variância Acumulada": variancia_acumulada
})

display(tabela_eigen.round(4))

# Critério de Kaiser
n_fatores_kaiser = int((tabela_eigen["Autovalor"] >= 1).sum())
print(f"Quantidade de fatores mantidos pelo critério de Kaiser: {n_fatores_kaiser}")

# Plot: autovalores
plt.figure(figsize=(8, 5))
plt.bar(tabela_eigen["Fator"], tabela_eigen["Autovalor"])
plt.axhline(1, linestyle="--", label="Critério de Kaiser: autovalor = 1")
plt.title("Autovalores por fator")
plt.xlabel("Fatores")
plt.ylabel("Autovalor")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

# Plot: variância explicada
plt.figure(figsize=(8, 5))
plt.bar(tabela_eigen["Fator"], tabela_eigen["Variância"])
plt.gca().yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
plt.title("Variância explicada por fator")
plt.xlabel("Fatores")
plt.ylabel("Variância explicada")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 5. Autovetores

Os autovetores indicam as direções dos fatores/componentes.  
Eles mostram como as variáveis originais se combinam para formar cada fator.

Resumo: enquanto o autovalor mede a quantidade de informação explicada, o autovetor indica a direção matemática do fator.

In [ ]:
#%% 5. Autovetores

autovetores = pca_full.components_.T

tabela_autovetores = pd.DataFrame(
    autovetores,
    index=imoveis_pca.columns,
    columns=[f"Autovetor {i+1}" for i in range(autovetores.shape[1])]
)

display(tabela_autovetores.round(4))

# Plot: heatmap simples dos autovetores
plt.figure(figsize=(9, 6))
plt.imshow(tabela_autovetores, vmin=-1, vmax=1)
plt.colorbar(label="Peso no autovetor")
plt.xticks(range(tabela_autovetores.shape[1]), tabela_autovetores.columns, rotation=45, ha="right")
plt.yticks(range(tabela_autovetores.shape[0]), tabela_autovetores.index)
plt.title("Autovetores dos fatores")

for i in range(tabela_autovetores.shape[0]):
    for j in range(tabela_autovetores.shape[1]):
        plt.text(j, i, f"{tabela_autovetores.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## 6. Cargas fatoriais

As cargas fatoriais representam a correlação entre cada variável original e cada fator gerado.

Resumo: elas ajudam a interpretar o conteúdo de cada fator.  
Por exemplo, se um fator tiver cargas altas em área, quartos, banheiros e vagas, ele pode representar uma dimensão de **porte/estrutura do imóvel**.

In [ ]:
#%% 6. Cargas fatoriais

# Para PCA com dados padronizados:
# carga fatorial = autovetor * raiz do autovalor

loadings = pca_full.components_.T * np.sqrt(pca_full.explained_variance_)

tabela_cargas = pd.DataFrame(
    loadings,
    index=imoveis_pca.columns,
    columns=[f"Fator {i+1}" for i in range(loadings.shape[1])]
)

display(tabela_cargas.round(4))

# Plot: cargas fatoriais dos fatores mantidos pelo Kaiser
fatores_mantidos = [f"Fator {i+1}" for i in range(n_fatores_kaiser)]

plot_cargas = tabela_cargas[fatores_mantidos]

x = np.arange(len(plot_cargas.index))
largura = 0.8 / max(1, len(fatores_mantidos))

plt.figure(figsize=(11, 5))
for idx, fator in enumerate(fatores_mantidos):
    plt.bar(x + idx * largura, plot_cargas[fator], width=largura, label=fator)

plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(x + largura * (len(fatores_mantidos)-1) / 2, plot_cargas.index, rotation=45, ha="right")
plt.title("Cargas fatoriais por variável")
plt.ylabel("Carga fatorial")
plt.legend()
plt.tight_layout()
plt.show()

# Loading plot: Fator 1 x Fator 2
if tabela_cargas.shape[1] >= 2:
    plt.figure(figsize=(7, 6))
    plt.scatter(tabela_cargas["Fator 1"], tabela_cargas["Fator 2"])

    for variavel in tabela_cargas.index:
        plt.text(
            tabela_cargas.loc[variavel, "Fator 1"] + 0.02,
            tabela_cargas.loc[variavel, "Fator 2"] + 0.02,
            variavel,
            fontsize=9
        )

    plt.axhline(0, linestyle="--")
    plt.axvline(0, linestyle="--")
    plt.title("Loading Plot — Fator 1 x Fator 2")
    plt.xlabel(f"Fator 1: {variancia[0]*100:.2f}% da variância")
    plt.ylabel(f"Fator 2: {variancia[1]*100:.2f}% da variância")
    plt.tight_layout()
    plt.show()

## 7. Comunalidades

A comunalidade mostra quanto da variância de cada variável original foi mantida pelos fatores selecionados.

Se mantivermos apenas alguns fatores, uma parte da informação fica de fora.

Interpretação:
- próximo de 1: a variável foi bem representada;
- próximo de 0: a variável perdeu muita informação.

Pergunta importante: **alguma variável deixou de ser bem representada pelos fatores mantidos?**

In [ ]:
#%% 7. Comunalidades com os fatores mantidos pelo critério de Kaiser

loadings_kaiser = tabela_cargas[fatores_mantidos]

comunalidades = (loadings_kaiser ** 2).sum(axis=1)

tabela_comunalidades = pd.DataFrame({
    "Comunalidade": comunalidades,
    "Variância não explicada": 1 - comunalidades
})

display(tabela_comunalidades.round(4))

# Plot: comunalidades
plt.figure(figsize=(9, 5))
plt.bar(tabela_comunalidades.index, tabela_comunalidades["Comunalidade"])
plt.axhline(0.70, linestyle="--", label="Referência: 0,70")
plt.ylim(0, 1.05)
plt.title("Comunalidades com os fatores selecionados")
plt.ylabel("Comunalidade")
plt.xticks(rotation=45, ha="right")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Extração dos fatores

A extração dos fatores gera uma nova pontuação para cada observação da amostra.

Em outras palavras, cada imóvel passa a ter um valor no Fator 1, Fator 2, Fator 3 etc.

Resumo: esses fatores são novas variáveis geradas a partir das variáveis originais padronizadas.

In [ ]:
#%% 8. Extração dos fatores para cada imóvel

pca_kaiser = PCA(n_components=n_fatores_kaiser)
fatores_extraidos = pca_kaiser.fit_transform(X_z)

fatores = pd.DataFrame(
    fatores_extraidos,
    columns=[f"Fator {i+1}" for i in range(n_fatores_kaiser)]
)

imoveis_com_fatores = pd.concat([imoveis.reset_index(drop=True), fatores], axis=1)

display(imoveis_com_fatores.head())

# Plot: distribuição dos fatores extraídos
n_cols = 2
n_rows = int(np.ceil(n_fatores_kaiser / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(10, 4 * n_rows))
axes = np.array(axes).reshape(-1)

for i, fator in enumerate(fatores.columns):
    axes[i].hist(fatores[fator], bins=12)
    axes[i].set_title(f"Distribuição do {fator}")
    axes[i].set_xlabel("Valor fatorial")
    axes[i].set_ylabel("Frequência")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()

# Plot: observações no plano Fator 1 x Fator 2, se existirem pelo menos 2 fatores
if n_fatores_kaiser >= 2:
    plt.figure(figsize=(7, 6))
    plt.scatter(imoveis_com_fatores["Fator 1"], imoveis_com_fatores["Fator 2"])
    plt.axhline(0, linestyle="--")
    plt.axvline(0, linestyle="--")
    plt.title("Imóveis no plano fatorial")
    plt.xlabel("Fator 1")
    plt.ylabel("Fator 2")
    plt.tight_layout()
    plt.show()

## 9. Scores fatoriais

Os scores fatoriais são os coeficientes que relacionam os fatores com as variáveis originais padronizadas em \(Z\).

Eles podem ser usados para demonstrar a fórmula do fator:

\[
F_1 = s_{11}Z_1 + s_{21}Z_2 + \cdots + s_{k1}Z_k
\]

Resumo: são os pesos usados para transformar as variáveis padronizadas em valores fatoriais.

In [ ]:
#%% 9. Scores fatoriais

# Em PCA, uma forma didática de obter coeficientes de score é:
# score = autovetor / raiz do autovalor
# Aqui calculamos para os fatores mantidos.

autovalores_kaiser = pca_kaiser.explained_variance_
autovetores_kaiser = pca_kaiser.components_.T

scores_fatoriais = autovetores_kaiser / np.sqrt(autovalores_kaiser)

tabela_scores = pd.DataFrame(
    scores_fatoriais,
    index=imoveis_pca.columns,
    columns=[f"Fator {i+1}" for i in range(n_fatores_kaiser)]
)

display(tabela_scores.round(4))

# Plot: scores fatoriais
x = np.arange(len(tabela_scores.index))
largura = 0.8 / max(1, n_fatores_kaiser)

plt.figure(figsize=(11, 5))
for idx, fator in enumerate(tabela_scores.columns):
    plt.bar(x + idx * largura, tabela_scores[fator], width=largura, label=fator)

plt.axhline(0, color="black", linewidth=0.8)
plt.xticks(x + largura * (n_fatores_kaiser - 1) / 2, tabela_scores.index, rotation=45, ha="right")
plt.title("Scores fatoriais por variável")
plt.ylabel("Score fatorial")
plt.legend()
plt.tight_layout()
plt.show()

# Exemplo da fórmula do Fator 1
formula_fator1 = "Fator 1 = " + " + ".join(
    [f"({tabela_scores.loc[var, 'Fator 1']:.4f})·Z_{var}" for var in tabela_scores.index]
)
print(formula_fator1)

## 10. Ranking: criação do indicador sintético

O ranking consolida os fatores em uma única métrica.

A ideia é ponderar cada fator pela sua variância explicada:

\[
Indicador_i = F_{1i} \cdot Var(F_1) + F_{2i} \cdot Var(F_2) + \cdots + F_{ki} \cdot Var(F_k)
\]

Assim, o fator com maior quantidade de informação recebe maior peso.

Depois, comparamos o indicador criado com o preço real do imóvel, que ficou fora da PCA.

In [ ]:
#%% 10. Ranking por soma ponderada dos fatores

variancias_kaiser = pca_kaiser.explained_variance_ratio_

imoveis_com_fatores["Indicador_PCA"] = 0

for i, fator in enumerate(fatores.columns):
    imoveis_com_fatores["Indicador_PCA"] += imoveis_com_fatores[fator] * variancias_kaiser[i]

# Ordenando imóveis pelo indicador
imoveis_ranking = imoveis_com_fatores.sort_values("Indicador_PCA", ascending=False).reset_index(drop=True)
imoveis_ranking["posicao_ranking"] = np.arange(1, len(imoveis_ranking) + 1)

display(
    imoveis_ranking[
        ["posicao_ranking", "id_imovel", "bairro", "tipo", "preco_venda_rs", "preco_m2_rs", "Indicador_PCA"]
    ].head(10)
)

# Correlação entre indicador e preço real
correlacao_preco = imoveis_com_fatores[["Indicador_PCA", "preco_venda_rs"]].corr().iloc[0, 1]
print(f"Correlação entre Indicador_PCA e preço real do imóvel: {correlacao_preco:.4f}")

# Plot: top 10 imóveis pelo indicador
top10 = imoveis_ranking.head(10).copy()
top10["rotulo"] = top10["id_imovel"].astype(str) + " - " + top10["bairro"]

plt.figure(figsize=(10, 5))
plt.barh(top10["rotulo"], top10["Indicador_PCA"])
plt.gca().invert_yaxis()
plt.title("Top 10 imóveis pelo Indicador PCA")
plt.xlabel("Indicador PCA")
plt.ylabel("Imóvel")
plt.tight_layout()
plt.show()

# Plot: indicador versus preço real
plt.figure(figsize=(7, 5))
plt.scatter(imoveis_com_fatores["Indicador_PCA"], imoveis_com_fatores["preco_venda_rs"])
plt.title(f"Indicador PCA x preço real do imóvel\nCorrelação = {correlacao_preco:.3f}")
plt.xlabel("Indicador PCA")
plt.ylabel("Preço de venda real")
plt.gca().yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, pos: f"R$ {x/1000:.0f} mil"))
plt.tight_layout()
plt.show()

## Interpretação final sugerida para o blog

A PCA foi aplicada com o objetivo de construir um indicador sintético para imóveis. Para isso, foram utilizadas apenas variáveis métricas relacionadas às características dos imóveis, deixando fora da análise o identificador, o preço de venda e o preço por metro quadrado. 

Após verificar os tipos das variáveis, observar possíveis outliers, analisar a matriz de correlação e aplicar o teste de Bartlett, foram extraídos os fatores principais. Em seguida, os fatores foram selecionados pelo critério de Kaiser, mantendo aqueles com autovalor maior ou igual a 1.

As cargas fatoriais permitiram interpretar o conteúdo representado por cada fator, enquanto as comunalidades indicaram se as variáveis originais continuaram bem representadas após a seleção dos fatores. Por fim, os fatores extraídos foram ponderados pelas respectivas variâncias explicadas, formando um único indicador. Esse indicador foi comparado com o preço real dos imóveis para verificar se a métrica criada capturava parte relevante do critério de valor.